# Gene-selection stability across outer training folds

Each selected-gene list in `results/results_<dataset>.csv` was produced by a selector that saw only
that outer fold's **training** rows, so this analysis never uses genes chosen with test data.
One selection per (repeat, fold) is used; rows for different classifiers are duplicates and are collapsed.

Indices: Kuncheva (equal panel sizes, chance-corrected), mean pairwise Jaccard, and Nogueira et al. (2018)
stability with a 95% asymptotic CI (handles variable sizes, e.g. GWO-free).

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
%matplotlib inline

In [ ]:
DATASET = os.environ.get("GWO_DATASET", "Leukemia_GSE9476")
RESULTS_DIR = ROOT / os.environ.get("GWO_RESULTS_DIR", "results")
DATA_PATH = ROOT / "data" / f"{DATASET}.csv"
K = int(os.environ.get("GWO_K", 30))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import stability, tables
from src.data import load_cumida

df = tables.read_results(tables.results_path(DATASET, RESULTS_DIR))
summary_csv = ROOT / "results" / "dataset_summary.csv"
if summary_csv.exists() and DATASET in set(pd.read_csv(summary_csv)["dataset"]):
    n_features = int(pd.read_csv(summary_csv).set_index("dataset").loc[DATASET, "n_features"])
    genes = load_cumida(DATA_PATH)[2] if DATA_PATH.exists() else None
else:
    X, _, genes = load_cumida(DATA_PATH)
    n_features = X.shape[1]
print(f"{DATASET}: {n_features} features; selectors {sorted(df['selector'].unique())}; k {sorted(df['k'].unique())}")

In [ ]:
table = stability.stability_table(df, n_features)
out = RESULTS_DIR / "tables"
out.mkdir(parents=True, exist_ok=True)
table.to_csv(out / f"stability_{DATASET}.csv", index=False)
(out / f"stability_{DATASET}.md").write_text(tables.to_markdown(table.round(3)) + "\n", encoding="utf-8")
table.round(3)

In [ ]:
path = stability.plot_stability(table, DATASET, RESULTS_DIR / "figures")
print(path)

## Overlap between selectors (mean per-fold Jaccard at k = K)

In [ ]:
overlap = stability.cross_selector_overlap(df, k=K, dataset=DATASET)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(overlap.to_numpy(dtype=float), cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(overlap)), overlap.columns, rotation=45, ha="right")
ax.set_yticks(range(len(overlap)), overlap.index)
for i in range(len(overlap)):
    for j in range(len(overlap)):
        v = overlap.iloc[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7,
                color="white" if v > 0.6 else "#0b0b0b")
fig.colorbar(im, ax=ax, label="Jaccard")
ax.set_title(f"Selector overlap, k={K}, {DATASET}", loc="left")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "figures" / f"selector_overlap_{DATASET}.png", dpi=200)
overlap.round(2)

## Most frequently selected genes per selector (k = K)

In [ ]:
freq_tables = {}
for sel in overlap.index:
    freq = stability.selection_frequency(stability.fold_selections(df, sel, k=K, dataset=DATASET), genes)
    freq_tables[sel] = freq.head(15)
top = pd.concat({s: f.reset_index() for s, f in freq_tables.items()}, names=["selector", "rank"])
top.to_csv(RESULTS_DIR / "tables" / f"top_genes_{DATASET}.csv")
top.groupby(level=0).head(5)

## Notes for the report

- [[Which selectors are most / least stable, and does stability rise with k?]]
- [[Does GWO's stability differ from the filter baselines at equal k?]]
- Repeated-CV folds overlap in training samples, so stability across folds is an upper bound on
  stability across independent cohorts.